# LeetCode 3735 — Lexicographically Smallest String After Reverse II bằng Python

- Link: https://leetcode.com/problems/lexicographically-smallest-string-after-reverse-ii/
- Độ khó: **Hard**
- Chủ đề: Chuỗi, so sánh từ điển, **hàm Z**, rolling hash + tìm nhị phân LCP, tỉa ứng viên (đoạn chạy ký tự nhỏ nhất)

> **Ghi chú:** Đây là bài **Premium**. Đề bài lấy từ bản doocs (leetcode.doocs.org); trang doocs không có mã nguồn, nên tên hàm `lexSmallest` là **suy luận** theo bài cùng loạt (3722 — bản I) — cần đối chiếu skeleton thật nếu có tài khoản Premium.

Skeleton (suy luận):

```python
class Solution:
    def lexSmallest(self, s: str) -> str:
```

## 1. Mục tiêu học

1. Liệt kê đúng không gian ứng viên: `2n` chuỗi (đảo `k` ký tự đầu hoặc `k` ký tự cuối, `1 ≤ k ≤ n`).
2. Viết lại hai họ ứng viên theo `t = rev(s)`: đảo đầu `P_k = t[n−k:] + s[k:]`, đảo cuối `S_m = s[:m] + t[:n−m]` (`m = n − k`).
3. So sánh hai ứng viên họ đảo cuối trong **`O(1)`** bằng **hàm Z** ⇒ tìm tốt nhất của họ này trong `O(n)`.
4. **Tỉa** họ đảo đầu: chỉ cần `k` là **cuối một đoạn chạy dài nhất** của ký tự nhỏ nhất.
5. So sánh với vét cạn `O(n²)` và với so sánh tổng quát bằng rolling hash (`O(n log n)` nhưng hằng số lớn trong Python).

## 2. Đề bài & ví dụ

Cho chuỗi `s` dài `n` (chữ thường). Thực hiện **đúng một** thao tác: chọn `k ∈ [1, n]` rồi **đảo `k` ký tự đầu** hoặc **đảo `k` ký tự cuối**. Trả về chuỗi nhỏ nhất theo thứ tự từ điển có thể thu được.

**Ví dụ 1:** `"dcab"` → `"acdb"` (đảo 3 ký tự đầu).

**Ví dụ 2:** `"abba"` → `"aabb"` (đảo 3 ký tự cuối).

**Ví dụ 3:** `"zxy"` → `"xzy"` (đảo 2 ký tự đầu).

**Ràng buộc:** `1 ≤ n ≤ 10^5`.

Lưu ý: `k = 1` không đổi chuỗi ⇒ bản thân `s` luôn là một ứng viên.

In [1]:
import random, time

examples = [("dcab", "acdb"), ("abba", "aabb"), ("zxy", "xzy")]
for s, exp in examples:
    print(s, "-> kỳ vọng", exp)

dcab -> kỳ vọng acdb
abba -> kỳ vọng aabb
zxy -> kỳ vọng xzy


## 3. Kiến thức nền

### 3.1. Hai họ ứng viên viết theo `t = rev(s)`

- Đảo `k` ký tự đầu: `rev(s[:k]) = t[n−k:]` ⇒ `P_k = t[n−k:] + s[k:]`.
- Đảo `k` ký tự cuối, đặt `m = n − k ∈ [0, n−1]`: `rev(s[m:]) = t[:n−m]` ⇒ `S_m = s[:m] + t[:n−m]`.

In [2]:
s = "dcab"; n = len(s); t = s[::-1]
for k in range(1, n + 1):
    m = n - k
    assert s[:k][::-1] + s[k:] == t[n - k:] + s[k:]
    assert s[:m] + s[m:][::-1] == s[:m] + t[:n - m]
print("P_k:", [t[n - k:] + s[k:] for k in range(1, n + 1)])
print("S_m:", [s[:m] + t[:n - m] for m in range(n)])

P_k: ['dcab', 'cdab', 'acdb', 'bacd']
S_m: ['bacd', 'dbac', 'dcba', 'dcab']


### 3.2. Hàm Z

`Z[i]` = độ dài tiền tố chung dài nhất của `a` và `a[i:]`. Tính trong `O(|a|)`. Với `a = t + '#' + s`, `Z[n + 1 + m]` = `lcp(t, s[m:])`; với `a = t`, `Z[L]` = `lcp(t, t[L:])`.

In [3]:
def zfunc(a):
    n = len(a); z = [0] * n
    if n: z[0] = n
    l = r = 0
    for i in range(1, n):
        if i < r:
            z[i] = min(r - i, z[i - l])
        while i + z[i] < n and a[z[i]] == a[i + z[i]]:
            z[i] += 1
        if i + z[i] > r:
            l, r = i, i + z[i]
    return z

print(zfunc("aabxaab"))

[7, 1, 0, 0, 3, 1, 0]


### 3.3. So sánh từ điển và tiền tố chung

`A < B` ⇔ tại vị trí khác nhau đầu tiên `A` có ký tự nhỏ hơn (hoặc `A` là tiền tố thật của `B`; ở đây mọi ứng viên cùng độ dài nên không xảy ra). Biết LCP là so sánh được ngay.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

**Họ đảo cuối `S_m`.** Hai ứng viên `m₁ < m₂` có chung `s[:m₁]`; phần còn lại là `A = t[:n−m₁]` và `B = s[m₁:m₂] + t[:n−m₂]`. Đặt `L = m₂ − m₁`:

- so `t[:L]` với `s[m₁:m₂]` — dùng `lcp(t, s[m₁:])` (hàm Z trên `t#s`);
- nếu bằng nhau, so `t[L:n−m₁]` với `t[:n−m₂]` — dùng `lcp(t[L:], t)` (hàm Z trên `t`).

⇒ so sánh `O(1)`, quét `m` một lượt giữ ứng viên tốt nhất.

**Họ đảo đầu `P_k`.** Ký tự đầu của `P_k` là `s[k−1]`. Gọi `c = min(s)`. Ứng viên tốt nhất toàn cục bắt đầu bằng `c` (vì `P_k` với `s[k−1] = c` luôn có), nên chỉ xét `k` với `s[k−1] = c`. Hơn nữa, trong một đoạn chạy `c` từ `b` đến `e−1` (`b > 0`), chọn `k = e` (cuối đoạn) cho nhiều chữ `c` đứng đầu nhất ⇒ tốt hơn mọi `k` bên trong đoạn. Và đoạn chạy **dài hơn** thắng đoạn ngắn hơn. Còn lại chỉ các đoạn chạy có độ dài lớn nhất.

## 5. Cách 1 — Brute Force: dựng cả `2n` ứng viên

In [4]:
def brute(s):
    n = len(s); best = None
    for k in range(1, n + 1):
        for cand in (s[:k][::-1] + s[k:], s[:n - k] + s[n - k:][::-1]):
            if best is None or cand < best:
                best = cand
    return best

for s, exp in examples:
    print(brute(s), "kỳ vọng", exp)

acdb kỳ vọng acdb
aabb kỳ vọng aabb
xzy kỳ vọng xzy


**Nút thắt:** `2n` chuỗi, mỗi chuỗi dựng và so `O(n)` ⇒ `O(n²)` ≈ `2·10^10` ký tự với `n = 10^5`.

## 6. Ý tưởng thuật toán / bước đột phá

1. **Họ đảo cuối:** `best_S` = tốt nhất trong `S_0..S_{n−1}`, so sánh `O(1)` bằng hàm Z ⇒ `O(n)`.
2. **Họ đảo đầu:** chỉ giữ `k = e` với `[b, e)` là đoạn chạy của `c = min(s)`, `b > 0`, có độ dài lớn nhất `M`. (Đoạn chạy bắt đầu tại `b = 0` cho `P_k = s`, đã có trong họ đảo cuối với `m = n − 1`.)
3. Đáp án = nhỏ nhất của `best_S` và các ứng viên `P_e` còn lại (so trực tiếp bằng chuỗi — Python so sánh bằng C).

## 7. Các cách tiếp theo

- **Cách 2:** so sánh tổng quát bằng **rolling hash + tìm nhị phân LCP** trên các ứng viên "hai mảnh" — `O(n log n)` nhưng hằng số lớn trong Python.
- **Cách 3:** **hàm Z** cho họ đảo cuối + **tỉa** họ đảo đầu (lời giải nộp).

### 7.1. Cách 2 — Rolling hash + tìm nhị phân LCP

In [5]:
def hash_solution(s):
    n = len(s); t = s[::-1]; MOD = (1 << 61) - 1; B = 131
    pw = [1] * (n + 1)
    for i in range(n): pw[i + 1] = pw[i] * B % MOD
    def pre(x):
        h = [0] * (n + 1)
        for i, ch in enumerate(x): h[i + 1] = (h[i] * B + ord(ch)) % MOD
        return h
    hs, ht = pre(s), pre(t)
    seg = lambda h, a, b: (h[b] - h[a] * pw[b - a]) % MOD
    def parts(c):                        # ứng viên = (chuỗi1, mảng hash1, bắt đầu1, dài1), (… mảnh 2)
        typ, k = c
        if typ == 0: return (t, ht, n - k, k), (s, hs, k, n - k)          # P_k
        return (s, hs, 0, n - k), (t, ht, 0, k)                          # đảo k ký tự cuối
    def H(c, L):
        (_, h1, a1, l1), (_, h2, a2, _) = parts(c)
        if L <= l1: return seg(h1, a1, a1 + L)
        return (seg(h1, a1, a1 + l1) * pw[L - l1] + seg(h2, a2, a2 + L - l1)) % MOD
    def ch(c, i):
        (x1, _, a1, l1), (x2, _, a2, _) = parts(c)
        return x1[a1 + i] if i < l1 else x2[a2 + i - l1]
    def less(c, d):
        lo, hi = 0, n
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if H(c, mid) == H(d, mid): lo = mid
            else: hi = mid - 1
        return lo < n and ch(c, lo) < ch(d, lo)
    best = (0, 1)
    for typ in (0, 1):
        for k in range(1, n + 1):
            if less((typ, k), best): best = (typ, k)
    (x1, _, a1, l1), (x2, _, a2, l2) = parts(best)
    return x1[a1:a1 + l1] + x2[a2:a2 + l2]

for s, exp in examples:
    print(hash_solution(s), "kỳ vọng", exp)

acdb kỳ vọng acdb
aabb kỳ vọng aabb
xzy kỳ vọng xzy


**Nhận xét:** đúng và `O(n log n)`, nhưng mỗi so sánh tốn ~17 lần tính hash hai mảnh — trong Python khoảng 6–7 giây với `n = 10^5` (đo ở mục 12). Ta cần cấu trúc của bài để làm nhanh hơn.

### 7.2. Cách 3 — Hàm Z + tỉa ứng viên (lời giải chính)

In [6]:
def best_suffix_family(s):
    n = len(s); t = s[::-1]
    zt = zfunc(t); zts = zfunc(t + "#" + s)

    def less(m2, m1):                         # S_{m2} < S_{m1} ?  (m1 < m2)
        L = m2 - m1
        l1 = min(zts[n + 1 + m1], L)          # lcp(t, s[m1:]) giới hạn L
        if l1 < L:
            return s[m1 + l1] < t[l1]         # B có s[m1+l1], A có t[l1]
        rem = n - m2
        l2 = zt[L]                            # lcp(t[L:], t)
        if l2 >= rem:
            return False                      # bằng nhau
        return t[l2] < t[L + l2]

    best = 0
    for m in range(1, n):
        if less(m, best):
            best = m
    return s[:best] + t[:n - best]

def z_prune(s):
    n = len(s); c = min(s)
    best = best_suffix_family(s)
    runs = []; i = 0
    while i < n:
        if s[i] == c:
            j = i
            while j < n and s[j] == c: j += 1
            if i > 0: runs.append((i, j))
            i = j
        else:
            i += 1
    if runs:
        M = max(e - b for b, e in runs)
        for b, e in runs:
            if e - b == M:
                cand = s[:e][::-1] + s[e:]
                if cand < best: best = cand
    return best

for s, exp in examples:
    print(z_prune(s), "kỳ vọng", exp)

acdb kỳ vọng acdb
aabb kỳ vọng aabb
xzy kỳ vọng xzy


## 8. TRACE từng bước

In [7]:
def trace_suffix_family(s):
    n = len(s); t = s[::-1]
    print("s =", s, "| t = rev(s) =", t)
    print("lcp(t, s[m:]) =", [zfunc(t + "#" + s)[n + 1 + m] for m in range(n)])
    print("lcp(t[L:], t) =", zfunc(t))
    best = 0
    for m in range(n):
        cand = s[:m] + t[:n - m]
        better = m > 0 and cand < s[:best] + t[:n - best]
        if better: best = m
        print(f"  m={m}: S_m = {cand}{'  <- tốt nhất mới' if better else ''}")
    print("tốt nhất họ đảo cuối:", s[:best] + t[:n - best])

trace_suffix_family("abba")

s = abba | t = rev(s) = abba
lcp(t, s[m:]) = [4, 0, 0, 1]
lcp(t[L:], t) = [4, 0, 0, 1]
  m=0: S_m = abba
  m=1: S_m = aabb  <- tốt nhất mới
  m=2: S_m = abab
  m=3: S_m = abba
tốt nhất họ đảo cuối: aabb


In [8]:
def trace_prefix_family(s):
    c = min(s); n = len(s)
    print("s =", s, "| ký tự nhỏ nhất c =", c)
    runs = []; i = 0
    while i < n:
        if s[i] == c:
            j = i
            while j < n and s[j] == c: j += 1
            runs.append((i, j)); i = j
        else: i += 1
    print("các đoạn chạy của c [b, e):", runs)
    for k in range(1, n + 1):
        if s[k - 1] == c:
            tag = "cuối đoạn" if any(e == k for _, e in runs) else "giữa đoạn"
            print(f"  k={k} ({tag}): P_k = {s[:k][::-1] + s[k:]}")

trace_prefix_family("baabaaab")
print("đáp án:", z_prune("baabaaab"), "| brute:", brute("baabaaab"))

s = baabaaab | ký tự nhỏ nhất c = a
các đoạn chạy của c [b, e): [(1, 3), (4, 7)]
  k=2 (giữa đoạn): P_k = ababaaab
  k=3 (cuối đoạn): P_k = aabbaaab
  k=5 (giữa đoạn): P_k = abaabaab
  k=6 (giữa đoạn): P_k = aabaabab
  k=7 (cuối đoạn): P_k = aaabaabb
đáp án: aaabaabb | brute: aaabaabb


## 9. Code nộp LeetCode

In [9]:
class Solution:
    def lexSmallest(self, s: str) -> str:
        n = len(s)
        t = s[::-1]

        def zfunc(a):
            m = len(a); z = [0] * m; z[0] = m
            l = r = 0
            for i in range(1, m):
                if i < r:
                    z[i] = min(r - i, z[i - l])
                while i + z[i] < m and a[z[i]] == a[i + z[i]]:
                    z[i] += 1
                if i + z[i] > r:
                    l, r = i, i + z[i]
            return z

        zt = zfunc(t)
        zts = zfunc(t + "#" + s)
        best_m = 0
        for m2 in range(1, n):                     # họ đảo cuối: S_m = s[:m] + t[:n-m]
            m1 = best_m
            L = m2 - m1
            l1 = min(zts[n + 1 + m1], L)
            if l1 < L:
                if s[m1 + l1] < t[l1]:
                    best_m = m2
            else:
                l2 = zt[L]
                rem = n - m2
                if l2 < rem and t[l2] < t[L + l2]:
                    best_m = m2
        best = s[:best_m] + t[:n - best_m]

        c = min(s)                                 # họ đảo đầu: chỉ cuối các đoạn chạy c dài nhất
        runs = []
        i = 0
        while i < n:
            if s[i] == c:
                j = i
                while j < n and s[j] == c:
                    j += 1
                if i > 0:
                    runs.append((i, j))
                i = j
            else:
                i += 1
        if runs:
            M = max(e - b for b, e in runs)
            for b, e in runs:
                if e - b == M:
                    cand = t[n - e:] + s[e:]
                    if cand < best:
                        best = cand
        return best

sol = Solution()
for s, exp in examples:
    got = sol.lexSmallest(s)
    print(s, "->", got, "✔" if got == exp else "✘")

dcab -> acdb ✔
abba -> aabb ✔
zxy -> xzy ✔


## 10. Vì sao thuật toán đúng

1. **Không gian ứng viên** = `{P_k}` ∪ `{S_m}`; `P_n = S_0 = rev(s)` và `S_{n−1} = s` (đảo 1 ký tự) nên `s` luôn là ứng viên.
2. **So sánh `S_{m₁}` và `S_{m₂}`** (`m₁ < m₂`) đúng như mục 4: hai mảnh liên tiếp, mỗi mảnh xác định bằng một giá trị LCP lấy từ hàm Z. Quét tuyến tính giữ ứng viên nhỏ nhất ⇒ `best_S` là nhỏ nhất họ đảo cuối.
3. **Tỉa họ đảo đầu.**
   - Có ứng viên bắt đầu bằng `c` (ví dụ `P_k` với `s[k−1] = c`) ⇒ ứng viên tối ưu bắt đầu bằng `c` ⇒ chỉ cần `P_k` với `s[k−1] = c`.
   - Với `k` trong đoạn chạy `[b, e)`, `b > 0`: `P_k = c^{k−b} · s[b−1] · …` với `s[b−1] ≠ c` nên `> c`. `P_e = c^{e−b} · s[b−1] · …` có `c` ở vị trí `k − b` ⇒ `P_e < P_k` khi `k < e`.
   - `P_e` bắt đầu bằng đúng `e − b` chữ `c` rồi tới ký tự `> c` ⇒ đoạn dài hơn cho chuỗi nhỏ hơn ⇒ chỉ giữ đoạn dài nhất.
   - Đoạn bắt đầu tại `b = 0`: `rev(c^k) = c^k` ⇒ `P_k = s`, đã có trong `best_S`.
4. Lấy `min` của các ứng viên còn lại ⇒ đáp án.

**Độ phức tạp:** hàm Z `O(n)`; họ đảo đầu còn `R ≤ n/(M+1)` ứng viên, mỗi ứng viên dựng `O(n)` bằng thao tác chuỗi trong C ⇒ xấu nhất `O(n²/M)` phép sao chép byte, thực tế ~2 s với `"ab" * 50000`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Vét cạn | `O(n²)` | `O(n)` | 2n chuỗi |
| 2. Hash + LCP | `O(n log n)` | `O(n)` | hằng số Python lớn |
| 3. Z + tỉa | `O(n)` + `O(R · n)` memcpy | `O(n)` | `R` = số đoạn chạy dài nhất |

## 12. Bộ test

In [10]:
METHODS = [brute, hash_solution, z_prune, lambda s: Solution().lexSmallest(s)]
for s, exp in examples:
    assert all(f(s) == exp for f in METHODS)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [11]:
# 12.2 ca biên
for s, exp in [("a", "a"), ("ba", "ab"), ("aaa", "aaa"), ("abc", "abc"), ("cba", "abc"), ("bab", "abb")]:
    assert all(f(s) == exp for f in METHODS), (s, [f(s) for f in METHODS])
print("Ca biên: OK")

Ca biên: OK


In [12]:
# 12.3 ngẫu nhiên (bảng chữ nhỏ để có nhiều đoạn chạy, nhiều hoà)
rng = random.Random(3735)
for _ in range(3000):
    s = "".join(rng.choice("abc"[:rng.randint(1, 3)]) for _ in range(rng.randint(1, 14)))
    b = brute(s)
    assert all(f(s) == b for f in METHODS[1:]), s
print("3000 test ngẫu nhiên: OK")

3000 test ngẫu nhiên: OK


In [13]:
# 12.4 test lớn: so thời gian Cách 2 và Cách 3
rng = random.Random(9)
cases = {"ngẫu nhiên a-z": "".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(10**5)),
         "toàn 'a'": "a" * 10**5,
         "'ab' lặp (xấu nhất cho tỉa)": "ab" * 50000}
for name, s in cases.items():
    t0 = time.time(); x = Solution().lexSmallest(s); t1 = time.time()
    print(f"{name}: Cách 3 mất {t1 - t0:.2f}s")
s = cases["ngẫu nhiên a-z"]
t0 = time.time(); y = hash_solution(s); t1 = time.time()
print(f"Cách 2 (hash) trên chuỗi ngẫu nhiên mất {t1 - t0:.2f}s, khớp: {y == Solution().lexSmallest(s)}")

ngẫu nhiên a-z: Cách 3 mất 0.07s
toàn 'a': Cách 3 mất 0.10s


'ab' lặp (xấu nhất cho tỉa): Cách 3 mất 0.51s


Cách 2 (hash) trên chuỗi ngẫu nhiên mất 3.73s, khớp: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ xét đảo tiền tố

In [14]:
def wrong_prefix_only(s):
    return min(s[:k][::-1] + s[k:] for k in range(1, len(s) + 1))

print("sai:", wrong_prefix_only("abba"), "| đúng:", brute("abba"))

sai: abba | đúng: aabb


### Lỗi 2 — Bắt buộc `k ≥ 2` (nghĩ "đúng một thao tác" phải làm đổi chuỗi)

In [15]:
def wrong_k_at_least_2(s):
    n = len(s)
    return min(min(s[:k][::-1] + s[k:], s[:n - k] + s[n - k:][::-1]) for k in range(2, n + 1))

print("sai:", wrong_k_at_least_2("abc"), "| đúng:", brute("abc"))

sai: acb | đúng: abc


### Lỗi 3 — Chọn `k` ở **đầu** đoạn chạy ký tự nhỏ nhất (thay vì cuối)

In [16]:
def wrong_run_start(s):
    n = len(s); c = min(s); best = best_suffix_family(s)
    for k in range(1, n + 1):
        if s[k - 1] == c and (k == 1 or s[k - 2] != c):        # SAI: chỉ k là đầu đoạn
            best = min(best, s[:k][::-1] + s[k:])
    return best

print("sai:", wrong_run_start("baab"), "| đúng:", brute("baab"))

sai: abab | đúng: aabb


### Lỗi 4 — Chỉ giữ **đoạn chạy dài nhất đầu tiên** (bỏ các đoạn cùng độ dài)

Tìm phản ví dụ bằng tìm kiếm ngẫu nhiên có seed:

In [17]:
def wrong_first_max_run(s):
    n = len(s); c = min(s); best = best_suffix_family(s)
    runs = []; i = 0
    while i < n:
        if s[i] == c:
            j = i
            while j < n and s[j] == c: j += 1
            if i > 0: runs.append((i, j))
            i = j
        else: i += 1
    if runs:
        M = max(e - b for b, e in runs)
        b, e = next(r for r in runs if r[1] - r[0] == M)        # SAI: chỉ đoạn đầu tiên
        best = min(best, s[:e][::-1] + s[e:])
    return best

rng = random.Random(11)
while True:
    s = "".join(rng.choice("abc") for _ in range(rng.randint(3, 10)))
    if wrong_first_max_run(s) != brute(s): break
print("phản ví dụ:", s, "| sai:", wrong_first_max_run(s), "| đúng:", brute(s))

phản ví dụ: bccacabb | sai: accbcabb | đúng: acaccbbb


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Một thao tác, chọn tham số `k`" ⇒ liệt kê họ ứng viên, rồi tìm cách **so sánh nhanh** hai ứng viên.
- Ứng viên có dạng "tiền tố của `A` + tiền tố của `B`" ⇒ **hàm Z** trên `B#A` và `B` cho LCP `O(1)`.
- Muốn nhỏ nhất theo từ điển ⇒ ký tự đầu phải là `min(s)` ⇒ tỉa mạnh ứng viên.
- Trong Python, so sánh/sao chép chuỗi chạy trong C rất nhanh — đôi khi tỉa + so trực tiếp nhanh hơn rolling hash "đúng lý thuyết".

## 15. Biến thể của bài

1. **Bản I** (`n ≤ 1000`): vét cạn `O(n²)`.
2. **Chỉ đảo tiền tố** hoặc **chỉ đảo hậu tố**.
3. **Đảo một đoạn bất kỳ `[l, r]`**: `l` là vị trí đầu tiên mà `s[l] > min(s[l:])`; chỉ cần chọn `r`.
4. **Lớn nhất** theo từ điển: đổi chiều so sánh (ký tự lớn nhất, đoạn chạy dài nhất của nó).

In [18]:
# Biến thể 3: đảo một đoạn bất kỳ — tối ưu và vét cạn
def best_any_reverse(s):
    n = len(s); suf_min = list(s)
    for i in range(n - 2, -1, -1): suf_min[i] = min(s[i], suf_min[i + 1])
    l = next((i for i in range(n) if s[i] > suf_min[i]), None)
    if l is None: return s
    return min(s[:l] + s[l:r + 1][::-1] + s[r + 1:] for r in range(l, n) if s[r] == suf_min[l])

def brute_any(s):
    n = len(s)
    return min(s[:l] + s[l:r + 1][::-1] + s[r + 1:] for l in range(n) for r in range(l, n))

rng = random.Random(15)
for _ in range(500):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(1, 10)))
    assert best_any_reverse(s) == brute_any(s)
print("Biến thể 3 OK:", best_any_reverse("dcab"))

Biến thể 3 OK: acdb


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Khi nào dùng |
|---|---|---|---|
| 1. Vét cạn | Dựng 2n chuỗi | `O(n²)` | Bản I, kiểm chứng |
| 2. Hash + LCP | So sánh tổng quát | `O(n log n)` | Ngôn ngữ nhanh (C++/Java) |
| 3. Z + tỉa | Cấu trúc hai mảnh + đoạn chạy | gần `O(n)` | **Python** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `lexSmallest("cab")` = ?

**Bài 2 (tính tay).** Liệt kê `S_m` cho `s = "bcab"` và chọn nhỏ nhất.

**Bài 3 (điền TODO).** Viết `largest_after_reverse(s)` — chuỗi **lớn nhất** theo từ điển.

**Bài 4 (tự viết lại).** Chỉ dùng hàm Z, viết `best_prefix_only(s)` cho họ đảo tiền tố khi `s` **không có ký tự lặp** (gợi ý: khi đó chỉ có một vị trí `c`).

**Bài 5 (mini challenge).** Đếm số giá trị `k` (tính cả hai kiểu thao tác, tức `2n` lựa chọn) cho ra đúng đáp án tối ưu.

In [19]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def largest_after_reverse(s):
    # TODO: đối xứng với lời giải (đổi chiều so sánh)
    pass

In [20]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def count_optimal_choices(s):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `c = 'a'` ở vị trí 1. (2) Đảo 2 ký tự đầu: `"acb"`. (3) So với họ đảo cuối (`"bac"`, …) ⇒ `"acb"`.

**Bài 2.** (1) `t = "bacb"`. (2) `S_0 = "bacb"`, `S_1 = "bbac"`, `S_2 = "bcba"`, `S_3 = "bcab"`. (3) Nhỏ nhất `"bacb"`.

**Bài 3.** (1) Ký tự lớn nhất `C = max(s)`. (2) Đoạn chạy dài nhất của `C`. (3) So sánh họ đảo cuối với chiều ngược lại — đơn giản nhất: vét cạn có tỉa rồi kiểm chứng.

**Bài 4.** (1) Không lặp ⇒ `c` xuất hiện một lần tại `p`. (2) Chỉ `k = p + 1`. (3) `P = t[n−p−1:] + s[p+1:]`.

**Bài 5.** (1) Tính đáp án `A`. (2) Đếm `k` sao cho ứng viên bằng `A` (vét cạn cho `n` nhỏ). (3) Chú ý `k = 1` hai kiểu đều cho `s`.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1, 2
assert Solution().lexSmallest("cab") == "acb" == brute("cab")
s2 = "bcab"; t2 = s2[::-1]
assert min(s2[:m] + t2[:4 - m] for m in range(4)) == "bacb"
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [22]:
# Bài 3, 4
def largest_after_reverse_sol(s):
    n = len(s)
    return max(max(s[:k][::-1] + s[k:], s[:n - k] + s[n - k:][::-1]) for k in range(1, n + 1))

def best_prefix_only_sol(s):
    n = len(s); t = s[::-1]; p = s.index(min(s))
    return t[n - p - 1:] + s[p + 1:]

rng = random.Random(4)
for _ in range(300):
    letters = list("abcdefgh"); rng.shuffle(letters)
    s = "".join(letters[:rng.randint(1, 8)])
    assert best_prefix_only_sol(s) == min(s[:k][::-1] + s[k:] for k in range(1, len(s) + 1))
print("Bài 3 ví dụ:", largest_after_reverse_sol("dcab"), "| Bài 4 OK")

Bài 3 ví dụ: dcba | Bài 4 OK


In [23]:
# Bài 5
def count_optimal_choices_sol(s):
    n = len(s); A = Solution().lexSmallest(s); cnt = 0
    for k in range(1, n + 1):
        cnt += (s[:k][::-1] + s[k:] == A) + (s[:n - k] + s[n - k:][::-1] == A)
    return cnt

print([(s, count_optimal_choices_sol(s)) for s in ["dcab", "abba", "aaa", "abc"]])
assert count_optimal_choices_sol("aaa") == 6

[('dcab', 1), ('abba', 1), ('aaa', 6), ('abc', 2)]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
t = s[::-1]
# Họ đảo cuối: S_m = s[:m] + t[:n-m], so S_m2 vs S_m1 bằng Z(t#s) và Z(t) -> O(1)
# Họ đảo đầu: chỉ k = e, [b,e) là đoạn chạy min(s) dài nhất, b > 0; P = t[n-e:] + s[e:]
ans = min(best_S, các P còn lại)
```

**Checklist:**

- [ ] Xét **cả hai** kiểu thao tác?
- [ ] `k = 1` (giữ nguyên `s`) là hợp lệ?
- [ ] Chọn `k` ở **cuối** đoạn chạy, giữ **mọi** đoạn chạy dài nhất?
- [ ] Chỉ số hàm Z trên `t + '#' + s` lệch `n + 1`?
- [ ] Đã so với vét cạn trên bảng chữ nhỏ?